In [8]:
# Cell 1 — Imports and load data

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.linalg as la
import warnings
warnings.filterwarnings('ignore')

from linearmodels.iv import IV2SLS
from statsmodels.tools import add_constant as sm_add_const

# ← 修改为你的实际路径
IN_PKL = r"C:\Users\asus\Desktop\acs_ssc_msub_v3.pkl"

df = pd.read_pickle(IN_PKL)
print(f"Loaded: {df.shape}")
print(f"Married share: {df['sscouple_mar'].mean():.3f}")

Loaded: (37907, 68)
Married share: 0.426


In [9]:
import pandas as pd

# Cell 2 — Prepare variables
# 用 errors='coerce' 防止偶尔出现的空值/脏数据导致整段代码崩溃
df["married"]        = pd.to_numeric(df["sscouple_mar"], errors='coerce')
df["msub_obs_k"]     = pd.to_numeric(df["msub_total_obs"], errors='coerce') / 1000
df["msub_hat_k"]     = pd.to_numeric(df["msub_total_hat"], errors='coerce') / 1000
df["legal_marriage"] = pd.to_numeric(df["staterecog_policy"], errors='coerce')
df["male"]           = pd.to_numeric(df["r_male"], errors='coerce')
df["any_children"]   = pd.to_numeric(df["c_anychildren"], errors='coerce')
df["n_children"]     = pd.to_numeric(df["c_children"], errors='coerce')
df["age_older"]      = pd.to_numeric(df["c_agemax"], errors='coerce')
df["age_diff"]       = pd.to_numeric(df["c_agediff"], errors='coerce')
df["edu_max"]        = pd.to_numeric(df["c_edumax"], errors='coerce')
df["edu_diff"]       = pd.to_numeric(df["c_edudiff"], errors='coerce')
df["same_race"]      = pd.to_numeric(df["c_racecomp"], errors='coerce')
df["medicaid_exp"]   = pd.to_numeric(df["medicaid_exp"], errors='coerce')
df["earn_split_obs"] = pd.to_numeric(df["c_incearn_split"], errors='coerce')
df["earn_split_hat"] = pd.to_numeric(df["hat_c_split"], errors='coerce')

# 核心修复点：直接利用已有的变量计算 1-5 次多项式，满足组员的回归模型需求
for i in range(1, 6):
    df[f"hat_earn{i}"]  = pd.to_numeric(df["hat_c_incearn"], errors='coerce') ** i
    df[f"hat_split{i}"] = pd.to_numeric(df["hat_c_split"], errors='coerce') ** i

df["year_fe"]  = df["year"].astype(str)
df["state_fe"] = df["statefip"].astype(str)

key_cols = ["married", "msub_obs_k", "msub_hat_k", "legal_marriage",
            "male", "any_children", "n_children", "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race", "medicaid_exp",
            "earn_split_obs", "earn_split_hat"]

# 剔除缺失值
df = df.dropna(subset=key_cols).copy()

print(f"Analysis sample: {len(df):,}")
print(f"Married share:   {df['married'].mean():.3f}  (paper: 0.432)")

Analysis sample: 37,907
Married share:   0.426  (paper: 0.432)


In [10]:
# Cell 3 — Helpers
def make_dummies(df, col):
    return pd.get_dummies(df[col], prefix=col, drop_first=True, dtype=float)

import numpy as np
from scipy import linalg as la

def drop_collinear(X, priority_cols=None):
    """
    ★ 修复版：
    1. 用原始矩阵的 numpy.linalg.matrix_rank 检测真实秩（能正确处理量级悬殊）
    2. 用列归一化后的带主元 QR 选择最优的线性无关列子集
    3. priority_cols 强制保留（const, msub_obs_k）
    """
    if priority_cols is None:
        priority_cols = []
    cols = X.columns.tolist()
    prio = [c for c in priority_cols if c in cols]
    rest = [c for c in cols if c not in prio]
    ordered = prio + rest  # priority 列优先，QR 更可能选中它们

    arr = X[ordered].values.astype(float)

    # Step 1: 在原始矩阵上检测真实秩
    rank = np.linalg.matrix_rank(arr)

    # Step 2: 列归一化后做带主元 QR，选出最优的 rank 个列
    col_norms = np.linalg.norm(arr, axis=0)
    col_norms[col_norms == 0] = 1.0
    arr_norm = arr / col_norms
    _, R, P = la.qr(arr_norm, pivoting=True, mode='economic')
    selected_idx = set(P[:rank].tolist())

    # Step 3: 强制把 priority 列换进来（踢掉一个非 priority 列）
    prio_pos = {ordered.index(c) for c in prio}
    for pi in prio_pos:
        if pi not in selected_idx:
            non_prio = [i for i in selected_idx if i not in prio_pos]
            if non_prio:
                selected_idx.discard(non_prio[-1])
            selected_idx.add(pi)

    keep = [ordered[i] for i in sorted(selected_idx)]
    return X[keep]

def run_spec(df, extra_controls=None, iv=False):
    extra_controls = extra_controls or []
    base = ["legal_marriage", "medicaid_exp", "male",
            "any_children", "n_children",
            "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race"]

    year_dums  = make_dummies(df, "year_fe")
    state_dums = make_dummies(df, "state_fe")
    X_cols = base + extra_controls

    X = pd.concat([df[X_cols], year_dums, state_dums], axis=1).astype(float)
    X = sm_add_const(X)
    y = df["married"]

    if not iv:
        # 1. OLS 模式
        X.insert(1, "msub_obs_k", df["msub_obs_k"].values)
        X = drop_collinear(X, priority_cols=["const", "msub_obs_k"])
        res = IV2SLS(y, X, None, None).fit(cov_type="robust")
    else:
        # 2. IV 模式
        endog = df[["msub_obs_k"]]
        instr = df[["msub_hat_k"]]
        combined = pd.concat([endog, X], axis=1)
        combined = drop_collinear(combined, priority_cols=["msub_obs_k"])
        endog  = combined[["msub_obs_k"]]
        X_exog = combined.drop(columns=["msub_obs_k"])
        res = IV2SLS(y, X_exog, endog, instr).fit(cov_type="robust")

    return res

def first_stage_info(res):
    try:
        fs    = res.first_stage
        indiv = fs.individual["msub_obs_k"]
        c     = float(indiv.params["msub_hat_k"])
        se    = float(indiv.std_errors["msub_hat_k"])
        p     = float(indiv.pvalues["msub_hat_k"])
        f     = float(fs.diagnostics.loc["msub_obs_k", "f.stat"])
        return c, se, p, f
    except Exception as e:
        print(f"first_stage_info error: {e}")
        return np.nan, np.nan, np.nan, np.nan

print("Helpers defined.")

Helpers defined.


In [11]:
# Cell 4 — Run all 6 columns
poly_cols = ([f"hat_earn{i}" for i in range(1, 6)] +
             [f"hat_split{i}" for i in range(1, 6)])

print("Col 1: OLS no income controls...")
col1 = run_spec(df, iv=False)

print("Col 2: IV  no income controls...")
col2 = run_spec(df, iv=True)

print("Col 3: OLS + earnings split...")
col3 = run_spec(df, extra_controls=["earn_split_obs"], iv=False)

print("Col 4: IV  + earnings split...")
col4 = run_spec(df, extra_controls=["earn_split_hat"], iv=True)

print("Col 5: OLS + poly + control function...")
col5 = run_spec(df, extra_controls=poly_cols, iv=False)

print("Col 6: IV  + poly + control function...")
col6 = run_spec(df, extra_controls=poly_cols, iv=True)

print("All columns estimated.")

Col 1: OLS no income controls...
Col 2: IV  no income controls...
Col 3: OLS + earnings split...
Col 4: IV  + earnings split...
Col 5: OLS + poly + control function...
Col 6: IV  + poly + control function...
All columns estimated.


In [12]:
# Cell 5 — Display table
def stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return "   "

def get_coef_se_p(res, name):
    if name not in res.params.index:
        return None, None, None
    return res.params[name], res.std_errors[name], res.pvalues[name]

def first_stage_info(res):
    try:
        indiv = res.first_stage.individual["msub_obs_k"]
        c  = float(indiv.params["msub_hat_k"])
        se = float(indiv.std_errors["msub_hat_k"])
        p  = float(indiv.pvalues["msub_hat_k"])
        f  = float(res.first_stage.diagnostics.loc["msub_obs_k", "f.stat"])
        return c, se, p, f
    except Exception as e:
        print(f"first_stage_info error: {e}")
        return np.nan, np.nan, np.nan, np.nan

results  = [col1, col2, col3, col4, col5, col6]
is_iv    = [False, True, False, True, False, True]
mean_dv  = df["married"].mean()

sep = "-" * 105

print("=" * 105)
print("TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)")
print("=" * 105)
hdr = f"{'':42}" + "".join(f"{'OLS' if not v else 'IV':>11}" for v in is_iv)
print(hdr)
print(sep)

def print_row(label, varname, paper_vals):
    # Coefficient row
    line = f"{label:42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if c is not None:
            line += f"  {c:7.3f}{stars(p)}"
        else:
            line += f"  {'—':>10}"
    print(line)
    # SE row
    line = f"{'':42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if se is not None:
            line += f"  ({se:6.3f})  "
        else:
            line += f"  {'':>10}"
    print(line)
    # Paper row
    print(f"  {'(paper)':40}" + "".join(f"  {v:>10}" for v in paper_vals))
    print()

print_row("Marriage subsidy ($1,000s)", "msub_obs_k",
          ["0.005***", "0.008***", "0.004***", "0.009*  ", "0.005***", "0.014***"])
print_row("Legal marriage", "legal_marriage",
          ["0.066***", "0.066***", "0.066***", "0.066***", "0.116***", "0.116***"])
print_row("State Medicaid expansion", "medicaid_exp",
          ["0.010   ", "0.010   ", "0.009   ", "0.010   ", "0.035***", "0.034***"])

print(sep)

# First stage
print(f"{'First stage coef':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  {c:7.3f}{stars(p)}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()

print(f"{'':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  ({se:6.3f})  ", end="")
    else:
        print(f"  {'':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'0.463***':>10}  {'—':>10}  {'0.408***':>10}  {'—':>10}  {'0.420***':>10}")
print()

print(f"{'First stage F-stat':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        _, _, _, f = first_stage_info(res)
        print(f"  {f:>10.1f}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'[474.7]':>10}  {'—':>10}  {'[221.0]':>10}  {'—':>10}  {'[261.3]':>10}")

print(sep)
print(f"{'Mean dep var':42}" + "".join(f"  {mean_dv:>10.3f}" for _ in results))
print(f"{'N':42}" + "".join(f"  {int(res.nobs):>10,}" for res in results))
print(f"  {'(paper N)':40}" + "".join(f"  {'37,234':>10}" for _ in results))
print("=" * 105)

TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)
                                                  OLS         IV        OLS         IV        OLS         IV
---------------------------------------------------------------------------------------------------------
Marriage subsidy ($1,000s)                    0.007***   -0.041***    0.005***   -0.008       0.005***   -0.074***
                                            ( 0.001)    ( 0.010)    ( 0.001)    ( 0.010)    ( 0.001)    ( 0.007)  
  (paper)                                     0.005***    0.008***    0.004***    0.009*      0.005***    0.014***

Legal marriage                                0.064***    0.063***    0.065***    0.064***           —           —
                                            ( 0.010)    ( 0.010)    ( 0.010)    ( 0.010)                          
  (paper)                                     0.066***    0.066***    0.066***    0.066***    0.116***    0.116***

State Medicaid expansion        

In [13]:
# Cell 6 — Elasticity
c6, _, _ = get_coef_se_p(col6, "msub_obs_k")
mean_msub = df["msub_obs_k"].mean()
elasticity = c6 * (mean_msub / mean_dv)
print(f"IV coef (col 6):     {c6:.4f}")
print(f"Mean subsidy ($k):   {mean_msub:.3f}")
print(f"Mean married:        {mean_dv:.3f}")
print(f"Implied elasticity:  {elasticity:.4f}  (paper: 0.011)")

IV coef (col 6):     -0.0735
Mean subsidy ($k):   0.461
Mean married:        0.426
Implied elasticity:  -0.0795  (paper: 0.011)


In [ ]:
# Cell 5 — Display table
def stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return "   "

def get_coef_se_p(res, name):
    if name not in res.params.index:
        return None, None, None
    return res.params[name], res.std_errors[name], res.pvalues[name]

def first_stage_info(res):
    try:
        indiv = res.first_stage.individual["msub_obs_k"]
        c  = float(indiv.params["msub_hat_k"])
        se = float(indiv.std_errors["msub_hat_k"])
        p  = float(indiv.pvalues["msub_hat_k"])
        f  = float(res.first_stage.diagnostics.loc["msub_obs_k", "f.stat"])
        return c, se, p, f
    except Exception as e:
        print(f"first_stage_info error: {e}")
        return np.nan, np.nan, np.nan, np.nan

results  = [col1, col2, col3, col4, col5, col6]
is_iv    = [False, True, False, True, False, True]
mean_dv  = df["married"].mean()

sep = "-" * 105

print("=" * 105)
print("TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)")
print("=" * 105)
hdr = f"{'':42}" + "".join(f"{'OLS' if not v else 'IV':>11}" for v in is_iv)
print(hdr)
print(sep)

def print_row(label, varname, paper_vals):
    # Coefficient row
    line = f"{label:42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if c is not None:
            line += f"  {c:7.3f}{stars(p)}"
        else:
            line += f"  {'—':>10}"
    print(line)
    # SE row
    line = f"{'':42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if se is not None:
            line += f"  ({se:6.3f})  "
        else:
            line += f"  {'':>10}"
    print(line)
    # Paper row
    print(f"  {'(paper)':40}" + "".join(f"  {v:>10}" for v in paper_vals))
    print()

print_row("Marriage subsidy ($1,000s)", "msub_obs_k",
          ["0.005***", "0.008***", "0.004***", "0.009*  ", "0.005***", "0.014***"])
print_row("Legal marriage", "legal_marriage",
          ["0.066***", "0.066***", "0.066***", "0.066***", "0.116***", "0.116***"])
print_row("State Medicaid expansion", "medicaid_exp",
          ["0.010   ", "0.010   ", "0.009   ", "0.010   ", "0.035***", "0.034***"])

print(sep)

# First stage
print(f"{'First stage coef':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  {c:7.3f}{stars(p)}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()

print(f"{'':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  ({se:6.3f})  ", end="")
    else:
        print(f"  {'':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'0.463***':>10}  {'—':>10}  {'0.408***':>10}  {'—':>10}  {'0.420***':>10}")
print()

print(f"{'First stage F-stat':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        _, _, _, f = first_stage_info(res)
        print(f"  {f:>10.1f}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'[474.7]':>10}  {'—':>10}  {'[221.0]':>10}  {'—':>10}  {'[261.3]':>10}")

print(sep)
print(f"{'Mean dep var':42}" + "".join(f"  {mean_dv:>10.3f}" for _ in results))
print(f"{'N':42}" + "".join(f"  {int(res.nobs):>10,}" for res in results))
print(f"  {'(paper N)':40}" + "".join(f"  {'37,234':>10}" for _ in results))
print("=" * 105)

TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)
                                                  OLS         IV        OLS         IV        OLS         IV
---------------------------------------------------------------------------------------------------------
Marriage subsidy ($1,000s)                    0.006***   -0.039***    0.004***   -0.033***    0.006***   -0.046***
                                            ( 0.001)    ( 0.011)    ( 0.001)    ( 0.010)    ( 0.001)    ( 0.012)  
  (paper)                                     0.005***    0.008***    0.004***    0.009*      0.005***    0.014***

Legal marriage                                0.064***    0.062***    0.065***    0.062***    0.065***    0.062***
                                            ( 0.010)    ( 0.010)    ( 0.010)    ( 0.010)    ( 0.010)    ( 0.010)  
  (paper)                                     0.066***    0.066***    0.066***    0.066***    0.116***    0.116***

State Medicaid expansion        